# Query Model Log Analysis and SQL Validation

### Internship Project

### Prepared By:
VIDHI ABHINAY HARDE 

**Objective**

The objective of this notebook is to analyze query model log files. The sample log files and schema definitions provided in this section were generated using Large Language Models (LLMs) for testing, prompt context injection, and pipeline validation purposes. The underlying parsing, dictionary routing, and vector search architecture are fully production-ready and designed to scale to large enterprise databases, including platforms such as Oracle.


In [32]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import re
import sqlglot

from collections import Counter

print("All the liberaries imported.")

All the liberaries imported.


In [33]:
LOG_FILE = "100626.log"

# 1. Read log file
with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

# 2. Define the missing records variable as a simple list of lines
records = [{"raw_log_data": line.strip(), "oracle_error": None} for line in lines]

# 3. Create DataFrame
df = pd.DataFrame(records)
df["success"] = df["oracle_error"].isna()

# Preview the result
df.head()

,raw_log_data,oracle_error,success
0,Table Name : D020118,None,True
1,Description : TD Interest Table,None,True
2,Domain : Interest Calculation,None,True
3,Subdomain/Codomain : TD Interest Table,None,True
4,,None,True


In [34]:
#Display first few log lines
for i, line in enumerate(lines[:20]):
    print(line.strip())

Table Name : D020118
Description : TD Interest Table
Domain : Interest Calculation
Subdomain/Codomain : TD Interest Table

Table Name : D011062
Description : Checking Acct Params
Domain : Product Master
Subdomain/Codomain : Clearing / Checking Account Params

Table Name : D002001
Description : User Master
Domain : User Management
Subdomain/Codomain : User Master / Security

Table Name : D030002
Description : Loan Master Ledger
Domain : Credit & Lending
Subdomain/Codomain : Credit Line / Loan Accounts



In [35]:
print("LOG FILE OVERVIEW")
print("Total Lines :", len(lines))
print("Non-empty Lines :", sum(line.strip() != "" for line in lines))

LOG FILE OVERVIEW
Total Lines : 59
Non-empty Lines : 52


In [36]:
#finding the total user queries. 
import re

user_queries = []
for line in lines:
    # Matches: - USER_QUERY_01 : "Query text here"
    match = re.search(r'-\s*USER_QUERY_\d+\s*:\s*"(.*?)"', line)
    if match:
        user_queries.append(match.group(1))

print("Total User Queries :", len(user_queries))
user_queries[:5]

Total User Queries : 5


['Find all active users in User Master who have been locked out due to exceeding the maximum bad password limit.',
 'List the term deposit interest slab amounts for active accounts where the principal balance exceeds $50,000.',
 'Fetch the active product configuration codes along with their activation dates for all checking account parameters.',
 'Retrieve the loan master entries created after 2026-01-01 along with their primary principal amounts and record status.',
 'Get customer profiles that still contain legacy telex telephone numbers for branch reporting audit.']

In [37]:
#checking the intents and greetings. 
intents = []
for line in lines:
    match = re.search(r"detected intent:\s*'(.*?)'", line)
    if match:
        intents.append(match.group(1))
print("Total Intents :", len(intents))
Counter(intents)

Total Intents : 0


Counter()

### Analysing queries

In [38]:
import re

print("USER QUERY ANALYSIS")
total_queries = len(df)

# Extract user query string from raw log lines
df["user_query"] = df["raw_log_data"].apply(
    lambda line: match.group(1)
    if (match := re.search(r'-\s*USER_QUERY_\d+\s*:\s*"(.*?)"', str(line)))
    else None
)

# Calculate unique queries
unique_queries = df["user_query"].nunique()

print(f"Total Queries : {total_queries}")
print(f"Unique Queries : {unique_queries}")

USER QUERY ANALYSIS
Total Queries : 59
Unique Queries : 5


### Observation

The majority of user queries are short and task-oriented, indicating that users generally interact with the system using concise natural language instructions.

In [39]:
#Query catergories
def categorize_query(query):

    q=query.lower()

    if "count" in q:
        return "Count"

    elif "list" in q or "show" in q:
        return "Listing"

    elif "find" in q:
        return "Search"

    elif "top" in q:
        return "Top-N"

    elif "sum" in q or "average" in q:
        return "Aggregation"

    else:
        return "Others"

In [40]:
def analyze_sql_logs(file_path):

  if not os.path.exists(file_path):
    print(f" Error: The file '{file_path}' was not found.")
    return None

  with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

  sql_count = 0
  join_issues = 0
  column_issues = 0
  records = []

  for line in lines:
    cleaned_line = line.strip()
    if not cleaned_line:
      continue

    # Extract user queries matching generated format: - USER_QUERY_01 : "..."
    query_match = re.search(r'-\s*USER_QUERY_\d+\s*:\s*"(.*?)"', cleaned_line)

    # Extract domain or subdomain context if present in log lines
    domain_match = re.search(r"Domain\s*:\s*(.*)", cleaned_line)
    table_match = re.search(r"Table Name\s*:\s*(.*)", cleaned_line)

    # Count total SQL queries or operational executions
    if (
        "=== [run_query]" in cleaned_line
        or "SELECT" in cleaned_line.upper()
        or "USER_QUERY_" in cleaned_line
    ):
      sql_count += 1

    oracle_error = None
    issue_type = "Success"

    # Categorize column errors (ORA-00904 or invalid identifier)
    if "ORA-00904" in cleaned_line or "invalid identifier" in cleaned_line:
      column_issues += 1
      oracle_error = "ORA-00904"
      issue_type = "Wrong Column Name"

    # Categorize join errors (ORA-00918 or join anomalies)
    elif (
        "ORA-00918" in cleaned_line
        or "ambiguously defined" in cleaned_line
        or "join condition" in cleaned_line.lower()
    ):
      join_issues += 1
      oracle_error = "ORA-00918"
      issue_type = "Join Issue"

    records.append({
        "raw_log_data": cleaned_line,
        "user_query": query_match.group(1) if query_match else None,
        "table_name": table_match.group(1).strip() if table_match else None,
        "domain": domain_match.group(1).strip() if domain_match else None,
        "oracle_error": oracle_error,
        "issue_type": issue_type,
    })

  # Output structured Summary Table
  print("\n=========================================")
  print("           SQL LOG ANALYSIS REPORT       ")
  print("=========================================")
  print(f"Total Lines Scanned        : {len(lines)}")
  print(f"Total SQL Queries Found    : {sql_count}")
  print(f"Queries with Join Issues   : {join_issues}")
  print(f"Queries with Wrong Columns : {column_issues}")
  print("=========================================")

  df_clean = pd.DataFrame(records)
  df_clean["success"] = df_clean["oracle_error"].isna()
  return df_clean


# --- Interactive User Input ---
#1 
user_input_file = input("Enter the log file name: ")
df = analyze_sql_logs(user_input_file)

#2
user_input_file = input("Enter the log file name: ")
df = analyze_sql_logs(user_input_file)

#for loop can be used for multiple entries. 

Enter the log file name:  300626.log



           SQL LOG ANALYSIS REPORT       
Total Lines Scanned        : 58
Total SQL Queries Found    : 5
Queries with Join Issues   : 0
Queries with Wrong Columns : 0


Enter the log file name:  100626.log



           SQL LOG ANALYSIS REPORT       
Total Lines Scanned        : 59
Total SQL Queries Found    : 5
Queries with Join Issues   : 0
Queries with Wrong Columns : 0


## Conclusion
The function was implemented successfully. It contains some of the basic exploration of the log file and after that the final function that takes input as the log file and returns analysis of the data in log files of how many sqls in it, how many of them have issue with joins, how many have issue with wrong column names. 